# **Morning conferences**

In [4]:
# Libraries.
import wget                   # library to download files.
import re                     # library to use regular expressions.
import glob                   # library to get files from a directory.
from bs4 import BeautifulSoup # library to parse HTML and XML documents.
import os                     # library to interact with the operating system.
from pathlib import Path      # library to interact with the file system.

### **We download a particular conference:**

In [5]:
# 30 September 2024.
url_conference_j = "https://amlo.presidente.gob.mx/30-09-24-version-estenografica-de-la-ultima-conferencia-de-prensa-del-presidente-andres-manuel-lopez-obrador/" # URL of the conference.
wget.download(url = url_conference_j, out = "./conference_j.txt") # Download the conference.

'./conference_j.txt'

### **Download page 2 of the conference listing:**

In [6]:
url_page_2 = "https://amlo.presidente.gob.mx/secciones/version-estenografica/page/2/" # URL of the page.
wget.download(url = url_page_2, out = "./page_2.txt")                                 # Download the page.

'./page_2.txt'

### **On this page, we search all urls**

In [7]:
file_path = './page_2.txt'
if os.path.isfile(file_path):                              # Verifies if the file exists and if it is a regular file.
    with open(file_path, "r", encoding = "utf-8") as file: # Opens the file using with.
        content = file.read()                              # Reads the content of the file.

    urls = re.findall(                                     # Regular expression to find URLs.
        r'(http|ftp|https)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-]?)',
        content
    )
    for url in urls:                                       # If we want to visualize the URLs.
        print(url)
else:
    print("The file does not exist.")

('http', '151.394.mwp.accessdomain.com', '/wp-content/themes/diplomat/favicon.ico')
('https', 'amlo.presidente.gob.mx', '/feed/')
('https', 'amlo.presidente.gob.mx', '/xmlrpc.php')
('https', 'amlo.presidente.gob.mx', '')
('https', 'amlo.presidente.gob.mx', '/wp-content/themes/diplomat/helper/capcha/image.php/')
('https', 'amlo.presidente.gob.mx', '/wp-content/themes/diplomat/')
('https', 'amlo.presidente.gob.mx', '/wp-admin/admin-ajax.php')
('https', 'amlo.presidente.gob.mx', '/feed/')
('https', 'amlo.presidente.gob.mx', '/comments/feed/')
('https', 'amlo.presidente.gob.mx', '/secciones/version-estenografica/feed/')
('https', 'amlo.presidente.gob.mx', '/wp-content/plugins/arqam/assets/style.css?ver=01c6fe169027d47c82db1c675442ee9b')
('https', 'fonts.googleapis.com', '/css?family=Roboto+Slab%7CRoboto%7CPT+Serif%7CDroid+Serif%7CMontserrat&#038')
('https', 'amlo.presidente.gob.mx', '/wp-content/plugins/tmm_content_composer/css/fontello.css?ver=01c6fe169027d47c82db1c675442ee9b')
('https', 

### **Let's download the 153 pages from AMLO and the 23 pages for Claudia Sheinbaum (until february 2025):**

>An exploration was made on the pages where the conferences of both presidents are listed and it was obtained that the number of pages in https://amlo.presidente.gob.mx/secciones/version-estenografica/ of the conferences of the former president is $153$ and the number of pages in https://www.gob.mx/presidencia/es/archivo/articulos?idiom=es&order=DESC&page=1 of the current president is $23$, to date (03/02/2025).
>
>With this, the total number of pages ($153+23=176$) were downloaded in ".txt" format, and were saved in the "./pages/" folder, with names "$i$", where "$i$" belongs to AMLO if $1\leq i \leq 153$ and to Claudia in the other case.

In [8]:
# URL de las paginas de AMLO y Claudia.
url_pages_AMLO    = "https://amlo.presidente.gob.mx/secciones/version-estenografica/page/"          # URL de las paginas de AMLO.
url_pages_CLAUDIA = "https://www.gob.mx/presidencia/es/archivo/articulos?idiom=es&order=DESC&page=" # URL de las paginas de Claudia.

num_pages_AMLO    = 153             # Numero de paginas de las conferencias de AMLO.
num_pages_CLAUDIA = 23              # Numero de paginas de las conferencias de Claudia.
PAGES             = "./pages/"      # Directorio donde se guardaran las paginas de ambos presidentes.
os.makedirs(PAGES, exist_ok = True) # Crear el directorio si no existe.

for i in range(1, num_pages_AMLO + num_pages_CLAUDIA + 1): # For each page.
    try:
        if i <= num_pages_AMLO:
            # Download the file.
            wget.download(
                url = f"{url_pages_AMLO}{str(i)}/",
                out = os.path.join(PAGES, f"{str(i)}.txt")
            )
        else:
            # Download the file.
            wget.download(
                url = f"{url_pages_CLAUDIA}{str(i - num_pages_AMLO)}",
                out = os.path.join(PAGES, f"{str(i)}.txt")
            )
    except Exception as e:
        # Handle errors gracefully.
        print(f"Error downloading page {i}: {e}")

### **Now we have to extract the URLs that interest us from each page.**

>Next, all the URLs of interest were obtained from the 176 .txt files. In this case, it was necessary to make a distinction between the pages corresponding to each president, since the regular expression for searching for URLs proposed in practice 1:
>
>'(http|ftp|https)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-]?)'
>
>did not work for the pages corresponding to Claudia, because these URLs were in short format. With some research on the web and exploring the .txt files, it was found that an alternative is to use the regular expression:
>
>'href="(/presidencia/es/articulos/[^"]+)"'
>
>Obtaining good results (1471 URLs in total) and saving all the URLs corresponding to morning conferences in the variable *urls_conf_set*.
>
>Note that *list comprehension* is used to find URLs containing "stenografica-de-la" for AMLO's links and "version-stenografica-conferencia" for Claudia's, since these URL fragments are distinctive of each one (several were experimented with and several manual reviews were made to verify that they worked well). In addition, this practice of *list_comprehension* is used in the same way on several occasions below.

In [9]:
urls_conf = []                 # Lista para almacenar URLs
files = glob.glob(f"{PAGES}*") # Se obtienen los archivos del directorio.

if not files:
    print("There are no files in the directory.")
else:
    for f_page in files:           # Se itera sobre todos los archivos. 
        if os.path.isfile(f_page): # Se verifica si es un archivo regular.
            
            # Se lee el contenido del archivo (aquí se hicieron algunos cambios respecto a la práctica 1).
            with open(f_page, "r", encoding = "utf-8") as file:
                content = file.read()

            urls = [ # Primero se hace con la expresión para URLs completas (las de AMLO).
                path for _, _, path in re.findall(
                    r'(http|ftp|https)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-]?)',
                    content
                ) if "estenografica-de-la" in path # Este pedazo de URL sólo aparece en las conferencias de AMLO.
            ]
            urls_conf.extend(urls) # Se agregan las URL's a la lista.

            urls = [ # Luego se hace con la expresión para URLs relativas (las de Claudia).
                path for path in re.findall(
                    r'href="(/presidencia/es/articulos/[^"]+)"', # Expresión regular para las URLs de las conferencias de Claudia.
                    content
                ) if "version-estenografica-conferencia" in path # Este pedazo de URL sólo aparece en las conferencias de Claudia.
            ]

            # Una vez que tenemos las URL'S, se agregan a la lista.
            urls_conf.extend(urls)

    urls_conf_set = list(set(urls_conf))             # Se eliminan las URL's repetidas.
    print("URL's:\n", urls_conf_set)                 # Vemos las URL's de las conferencias.
    print("URL's Diferentes:\n", len(urls_conf_set)) # El número de URL's diferentes.

URL's:
 ['/19-06-23-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/26-08-21-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/13-09-24-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/18-02-21-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/17-06-22-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/26-02-24-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/12-01-21-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/12-01-24-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador/', '/11-12-20-version-estenografica-de-la-conferencia-de-prensa-ma

>In this part, the morning conferences of both presidents are downloaded, again, making a distinction between the links of each one and they are saved in the "./stenographies/" directory. It was obtained that there are $1350$ of AMLO's conferences and $101$ of Claudia's. Note that the base URL is different for each president.

In [10]:
BASE_URL_A = "https://amlo.presidente.gob.mx"   # URL base para las conferencias de AMLO.
BASE_URL_C = "https://www.gob.mx/"              # URL base para las conferencias de Claudia.
STENOGRAPHIES_DIR = "./stenographies/"          # Directorio donde se guardaran las estenografías.
os.makedirs(STENOGRAPHIES_DIR, exist_ok = True) # Crear el directorio si no existe.
n_AMLO = 0                                      # Contador para las conferencias de AMLO.
n_CLAUDIA = 0                                   # Contador para las conferencias de Claudia.

# Se descargan las estenografías.
for path in urls_conf_set:
    if "estenografica-de-la" in path: # Condicional para las conferencias de AMLO.
        try:
            # Descargamos el archivo.
            wget.download(
                url = f"{BASE_URL_A}{path}",
                out = os.path.join(STENOGRAPHIES_DIR, path.replace("/","-")) # Reeplazamos '/' con '-' en el nombre del archivo.
            )
            n_AMLO += 1 # Se incrementa el contador.
        except Exception as e:
            # Se manejan los errores.
            print(f"Failed to download {path}: {e}")
    elif "version-estenografica-conferencia" in path: # Condicional para las conferencias de Claudia.
        try:
            # Descargamos el archivo.
            wget.download(
                url = f"{BASE_URL_C}{path}",
                out = os.path.join(STENOGRAPHIES_DIR, path.replace("/","-")) # Reeplazamos '/' con '-' en el nombre del archivo.
            )
            n_CLAUDIA += 1 # Se incrementa el contador.
        except Exception as e:
            # Handle errors gracefully.
            print(f"Failed to download {path}: {e}")

print("Estenografías de AMLO   :", n_AMLO)    # Número de estenografías de AMLO.
print("Estenografías de Claudia:", n_CLAUDIA) # Número de estenografías de Claudia.

Estenografías de AMLO   : 1370
Estenografías de Claudia: 101


However, these documents are full of html garbage. We only need the information from the morning press conferences. The way to do it depends on each application, here is one proposed using the $\textit{bs4}$ library.

In [11]:
STENOGRAPHIES_DIR = "./stenographies/"                  # Directorio de las estenografías.
CLEANED_STENOGRAPHIES_DIR = "./cleaned_stenographies/"  # Directorio donde se guardaran las estenografías limpias.
os.makedirs(CLEANED_STENOGRAPHIES_DIR, exist_ok = True) # Crear el directorio si no existe.
files = glob.glob(f"{STENOGRAPHIES_DIR}*")              # Se obtienen los archivos del directorio.

if not files:
    print("There are no files in the directory.")
else:
    for f_page in files: # Se itera sobre todos los archivos.
        try:
            # Se utiliza BeautifulSoup para parsear el HTML.
            with open(f_page, "r", encoding = "utf-8") as file:
                soup = BeautifulSoup(file.read(), "html.parser")

            # Se obtiene el texto de la estenografía.
            output_file = os.path.join(CLEANED_STENOGRAPHIES_DIR, Path(f_page).name)

            # Se escribe el texto en un archivo.   
            with open(output_file, "w", encoding = "utf-8") as output:
                output.write(soup.get_text())

        except Exception as e:
            print(f"Error processing {f_page}: {e}")

>Finally, remember that all the conference files had $4$ dates at the time of searching for them, except for 5, which did not contain any. The first date to appear was used to name the corresponding file.
>
>However, this did not work for Claudia's conferences since the format of the dates contained in these files did not match those of AMLO. So a lot of digging had to be done among these files, concluding that the dates that appeared, corresponding to the day of the conference, were written in the form:
>
>7 de diciembre de 2024
>
>In addition, since she has given lectures from October 2024 to March 2025, it was decided to use the regular expression
>
>'\b(0?[1-9]|[12]\d|30|31) of (October|November|December|January|February|March) of (2024|2025)\b'
>
>Obtaining results that match quite well with manual explorations. Fortunately, of all the dates found in each of Claudia's conferences, the first one indicated the main date of the conference, so that was taken as the file name.
>
>Finally, observing the format in which Claudia's conference dates came, the function *normalize2()* was created, which converts a date in the format ('day', 'month', 'year') to the format 'YYYY-MM-DD'.

In [12]:
def normalize1(str_date: str) -> str: 
    day, month, year = str_date.split(".") # Split the date string.
    return f"20{year}-{month}-{day}"       # Return the normalized date string.
def normalize2(str_date: tuple) -> str:
    meses = {
        "enero"     : "01",
        "febrero"   : "02",
        "marzo"     : "03",
        "abril"     : "04",
        "mayo"      : "05",
        "junio"     : "06",
        "julio"     : "07",
        "agosto"    : "08",
        "septiembre": "09",
        "octubre"   : "10",
        "noviembre" : "11",
        "diciembre" : "12"
    }
    day, month, year = str_date
    n_month = meses.get(month.lower())        # Convertir nombre del month a número
    return f"{year}-{n_month}-{int(day):02d}" # Formatear con ceros a la izquierda

In [13]:
CLEANED_STENOGRAPHIES_DIR = "./cleaned_stenographies/" # Directorio donde se guardardan las estenografías limpias.
FINAL_CORPUS = "./final_corpus/"                       # Directorio final donde se guardaran las estenografías normalizadas.
os.makedirs(FINAL_CORPUS, exist_ok = True)             # Crear el directorio si no existe.
files = glob.glob(f"{CLEANED_STENOGRAPHIES_DIR}*")     # Se obtienen los archivos del directorio.

if not files:
    print("No files found in the directory.")
else:
    for f_page in files: # Iteramos sobre todos los archivos.
        try:
            with open(f_page, "r", encoding = "utf-8") as file: # Se lee el contenido del archivo.
                text = file.read()

            # Se intenta encontrar fechas en formato "dd.mm.yy" (el formato de AMLO).
            dates_in_file = re.findall(r'\b\d{2}\.\d{2}\.\d{2}\b', text)
            if dates_in_file:
                normalized_date = normalize1(dates_in_file[0]) # Se usa normalize1 en este caso.
            else:
            # Si no se encuentra ninguna fecha en el primer formato, se intenta encontrar fechas en el segundo formato.
                dates_in_file = re.findall(
                    r'\b(0?[1-9]|[12]\d|30|31) de (octubre|noviembre|diciembre|enero|febrero|marzo) de (2024|2025)\b',
                    text
                )
                if dates_in_file:
                        normalized_date = normalize2(dates_in_file[0]) # Se usa normalize2 en este caso.
                else:
                    print(f"Invalid date format in: {f_page}")
                    continue  # Si no encuentra ninguna fecha, pasa al siguiente archivo

            # Creamos el nombre del archivo de salida con el formato "YYYY-MM-DD.txt"
            normalized_file = os.path.join(FINAL_CORPUS, normalized_date)

            # Buscar un nombre disponible (_2, _3, etc.)
            if not os.path.isfile(normalized_file):
                output_file = normalized_file
            elif not os.path.isfile(normalized_file + "_2"):
                print(f"File already exists: {normalized_file}")
                output_file = normalized_file + "_2"
            elif not os.path.isfile(normalized_file + "_3"):
                print(f"File already exists again: {normalized_file}")
                output_file = normalized_file + "_3"
            else:
                print(f"File already exists 3 times: {normalized_file}")
                continue

            # Guardamos el texto normalizado en un archivo.
            with open(output_file, "w", encoding = "utf-8") as output:
                output.write(text)

        except Exception as e:
            print(f"Error processing {f_page}: {e}")

Invalid date format in: ./cleaned_stenographies/-13-05-2022-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador-desde-nuevo-leon-
Invalid date format in: ./cleaned_stenographies/-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador-67-
File already exists: ./final_corpus/2019-01-19
Invalid date format in: ./cleaned_stenographies/-29-0920-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador-
File already exists again: ./final_corpus/2019-01-19
Invalid date format in: ./cleaned_stenographies/-08-1020-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador-
File already exists: ./final_corpus/2023-02-23
Invalid date format in: ./cleaned_stenographies/-28-06-2019-version-estenografica-de-la-conferencia-de-prensa-matutina-del-presidente-andres-manuel-lopez-obrador-
File already exists: ./final_corpus

>The results obtained are the following:
>- There were five invalid formats corresponding to AMLO conferences, which did not contain any date in the proposed format, so we went $1471$ total conferences, with $1365$ for AMLO and $101$ for Claudia.
>- In addition, on 9 occasions there was more than one conference on the same day, all of them by AMLO. In fact, on January 19, 2019, there were three conferences.